In [1]:
from langchain_community.document_loaders import(
    PyPDFLoader,
    PyMuPDFLoader
)

e:\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain_community.document_loaders import TextLoader

# loading single text file
loader= TextLoader("module1_searchable.txt",encoding="utf-8")

documents = loader.load()
print(type(documents))
# print(documents)

<class 'list'>


In [9]:
# from langchain_community.document_loaders import DirectoryLoader

# # Load all the text files from the directory
# dir_loader = DirectoryLoader(
#     "data/text_files",
#     glob="**/*.txt",  # patter to match files
#     loader_cls=TextLoader, # loader class to sue
#     loader_kwargs={'encoding':'utf-8'},
#     show_progress=True
# )
# documents = dir_loader.load()
# print(f"loaded ({len(documents)}) documents")
# for i,doc in enumerate(documents):
#     print(f"Docuemnt {i+1}")
#     print(f"source : {doc.metadata['source']}")
#     print(f" Length : {len(doc.page_content)} characters")


In [10]:
text = documents[0].page_content

In [12]:
# text

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Recursive character text splitter")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""],
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]} ...")

Recursive character text splitter
created 799 chunks
First chunk: ﻿Student Material -
Professional.Series

EC-Councll
Official Curricula

FC-Gouncil C EH" Certified E ...


In [15]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader


# Load environment variables
load_dotenv()

python-dotenv could not parse statement starting at line 7


True

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize a simple embedding model(no api key is needed)
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 280.31it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [32]:
### load the embedding models
import os
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

python-dotenv could not parse statement starting at line 7


In [18]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [19]:
# step-2: Dense Retriver (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(documents,embedding_model)
dense_retriver = dense_vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 397.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Sparse Regtirver(BM25)
sparse_retriver = BM25Retriever.from_documents(documents)
sparse_retriver.k = 3 # top k document to retriver


In [22]:
from langchain_classic.retrievers import EnsembleRetriever

# step4 : combine with ensemble Retriver 
hybrid_retriver = EnsembleRetriever(
    retrievers = [dense_retriver,sparse_retriver],
    weight = [0.7,0.3]
)

In [23]:
hybrid_retriver

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000143F658ED50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000143F427B890>, k=3)], weights=[0.5, 0.5])

In [24]:
# Step 5: Query and get results
query = "A company has deployed a web application that allows users to upload profile pictures. As an ethical hacker, how would you test whether this file upload feature could be exploited to execute malicious code on the server? What security controls should you expect to be in place to prevent such an attack?"
results = hybrid_retriver.invoke(query)

# step 6 : print results
for i,doc in enumerate(results):
    print(f"\nDocument {i+1}:\n{doc.page_content}")


Document 1:
﻿Student Material -
Professional.Series

EC-Councll
Official Curricula

FC-Gouncil C EH" Certified Ethical Hacker



Ethical Hacking and
Countermeasures

gf

>q } y
» ~® dS
NN,

y, Version 13



Copyright © 2024 by EC-Council. All rights reserved. Except as permitted under the Copyright Act of 1976, no part
of this publication may be reproduced or distributed in any form or by any means, or stored in a database or retrieval
system, without the prior written permission of the publisher, with the exception that the program listings may be
entered, stored, and executed in a computer system, but may not be reproduced for publication without the prior
written permission of the publisher, except in the case of brief quotations embodied in critical reviews and certain
other noncommercial uses permitted by copyright law. For permission requests, write to EC-Council, addressed
“Attention: EC-Council,” at the address below:

EC-Council New Mexico
101C Sun Ave NE
Albuquerque, NM 8710

In [25]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

In [26]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [33]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")


llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
    )
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001438D308590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001438D33D5B0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [29]:
# Create stuff Document Chain
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

# Create full RAG chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriver,combine_docs_chain=document_chain)

In [35]:
# step 9 : Ask a question
query={"input":"What is the main goal of footprinting in ethical hacking?"}
response = rag_chain.invoke(query)

# step 10 : Output
print("Answer:\n",response["answer"])
print("\n Source Documents")
for i,doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01km0tm6cvf5qafcb7ytrqhb29` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 58317, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}